### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [2]:
from langchain.chat_models import init_chat_model
model = init_chat_model(model="granite4.1:3b", model_provider="ollama")
response = model.invoke("What is the capital of France?")
print(response)

content='The capital of France is Paris.' additional_kwargs={} response_metadata={'model': 'granite4.1:3b', 'created_at': '2026-05-24T19:01:06.7721392Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4607183800, 'load_duration': 3677027000, 'prompt_eval_count': 15, 'prompt_eval_duration': 347933300, 'eval_count': 8, 'eval_duration': 503200000, 'logprobs': None, 'model_name': 'granite4.1:3b', 'model_provider': 'ollama'} id='lc_run--019e5b5c-b252-7100-ac76-041220e7fe51-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 15, 'output_tokens': 8, 'total_tokens': 23}


In [5]:
from langchain.tools import tool
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    # In a real implementation, you would call a weather API here.
    return f"The current weather in {location} is sunny with a temperature of 25°C."

model_with_weather = model.bind_tools([get_weather])

In [16]:
response = model_with_weather.invoke("What is the current weather in New York?")
# content is not available as tool responded.
print(response.content)

#tool message is available in tool_calls
for tool_call in response.tool_calls:
    print(f"Tool called: {tool_call['name']}")
    print(f"Tool input: {tool_call['args']}")


Tool called: get_weather
Tool input: {'location': 'New York'}


### Tool execution loops.

In [22]:
messages = [{"role": "user", "content": "What is the current weather in Bangalore?"}]
# Step 1 - Call the model and capture the ai message it returns. 
aimessage = model_with_weather.invoke(messages)
messages.append(aimessage)

# Step 2 - Check if there are any tool calls in the ai message
if aimessage.tool_calls:
    for tool_call in aimessage.tool_calls:
        print(f"Tool called: {tool_call['name']}")
        print(f"Tool args: {tool_call['args']}")
        if tool_call["name"] == "get_weather":
            tool_result = get_weather.invoke(tool_call)
            messages.append(tool_result) 

#Print each message in the messages list.
for amsg in messages:
    print(amsg)

#Step 3 = Call the model again with the updated messages which now includes the tool response.
final_response = model_with_weather.invoke(messages)
print("Final response from model:")
print(final_response.content)


Tool called: get_weather
Tool args: {'location': 'Bangalore'}
{'role': 'user', 'content': 'What is the current weather in Bangalore?'}
content='' additional_kwargs={} response_metadata={'model': 'granite4.1:3b', 'created_at': '2026-05-24T19:24:46.3467317Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1863439400, 'load_duration': 70224600, 'prompt_eval_count': 175, 'prompt_eval_duration': 86809200, 'eval_count': 21, 'eval_duration': 1529767900, 'logprobs': None, 'model_name': 'granite4.1:3b', 'model_provider': 'ollama'} id='lc_run--019e5b72-6645-7e52-aaba-ae6058dda888-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Bangalore'}, 'id': '3c45884c-ce46-4a7d-8358-d8027334b205', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 175, 'output_tokens': 21, 'total_tokens': 196}
content='The current weather in Bangalore is sunny with a temperature of 25°C.' name='get_weather' tool_call_id='3c45884c-ce46-4a7d-8358-d8027334b205'
Final response from m